# Week 3 · Notebook 2 — The reveal, and your first deep RL agent

First, a secret worth pausing on. Then a real reinforcement-learning agent (PPO)
that allocates across the whole universe — trained honestly, judged on data it
never saw.

Two functions you build:
1. `action_to_weights` — turn the agent's raw output into a real portfolio.
2. `evaluate_agent` — run the trained agent and score it (reusing `report`).

## 1. The reveal — you already built the environment

An RL agent learns inside an *environment*: it calls `reset()` to start and
`step(action)` to act. Watch what `PortfolioEnv.step` actually does:

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np, json
import matplotlib.pyplot as plt
os.makedirs('dashboard/data', exist_ok=True)
from tradinglab.env import PortfolioEnv
import inspect
src = inspect.getsource(PortfolioEnv.step)
print(src[src.index('# --- CORE'):src.index('# --- REWARD')])

> It calls `self.sim.step_return(...)` — the **exact** portfolio-accounting
> function your week-1 backtester calls. Strip away the `reset`/`step` wrapper that
> reinforcement learning needs, and the environment *is* the simulator you built in
> week 1. The agent just replaces the hand-coded strategy with a learned one.

## 3. Function — `evaluate_agent`

Run a trained agent through an environment once, collecting the daily portfolio and
benchmark returns, and return a `report()`-compatible dict.

**In:** `model`, `env`. **Out:** dict with `portfolio`, `benchmark`,
`portfolio_returns`, `benchmark_returns`.
**Hint:** loop `model.predict(obs)` → `env.step(action)`, collect
`info['portfolio_return']` and `info['benchmark_return']`.
**Done when:** the check returns curves of the right length.

In [ ]:
def evaluate_agent(model, env):
    # evaluation must cover the full day range, even if training randomized the
    # episode start -- otherwise the curve and the dates don't line up
    was_random = getattr(env, 'randomize_start', False)
    env.randomize_start = False
    obs, _ = env.reset(); done = False
    t0 = env.t
    port, bench = [], []
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
        port.append(info['portfolio_return']); bench.append(info['benchmark_return'])
    env.randomize_start = was_random
    port, bench = np.array(port), np.array(bench)
    # same date convention as every other notebook: a return computed AT day t
    # is realized ON day t+1 -- see week 1's backtester for the same offset
    dates = env.feed.dates[t0 + 1 : t0 + 1 + len(port)]
    return {'portfolio': np.cumprod(1+port), 'benchmark': np.cumprod(1+bench),
            'portfolio_returns': port, 'benchmark_returns': bench, 'dates': dates}


## 4. Train PPO — honestly, with a train/test split
Train on the early period, evaluate on the later one the agent never saw. (Small
run here; use `notebooks/colab_train.ipynb` on free Colab/Kaggle for longer.)

In [ ]:
from tradinglab.data_feed import DataFeed
from tradinglab.train import make_envs, split_day
from tradinglab.report import report
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

# feed = DataFeed.from_dir('data/egx')
feed = DataFeed.from_dir('data/egx', symbols=['COMI', 'HRHO', 'TMGH', 'SWDY', 'FWRY', 'ABUK'])

split = split_day(feed, 0.7)
# train_env, test_env = make_envs(feed, split, reward='excess', top_k=2)
train_env, test_env = make_envs(feed, split, reward='excess', top_k=2, randomize_start=True)

model = PPO('MlpPolicy', train_env, verbose=0, n_steps=512, seed=0)


class LossHistoryCallback(BaseCallback):
    """Pulls loss values out of PPO's internal logger after each training
    update, so you get a real loss curve to plot -- not just final performance."""
    def __init__(self):
        super().__init__()
        self.value_loss = []
        self.policy_loss = []
        self.entropy_loss = []

    def _on_step(self):
        return True

    def _on_rollout_end(self):
        logs = self.model.logger.name_to_value
        if "train/value_loss" in logs:
            self.value_loss.append(logs["train/value_loss"])
            self.policy_loss.append(logs["train/policy_gradient_loss"])
            self.entropy_loss.append(logs["train/entropy_loss"])

loss_cb = LossHistoryCallback()
model = PPO('MlpPolicy', train_env, verbose=0, n_steps=512, seed=0)
model.learn(total_timesteps=50000, callback=loss_cb)

plt.figure(figsize=(9,4))
plt.plot(loss_cb.value_loss, label='value loss')
plt.plot(loss_cb.policy_loss, label='policy loss')
plt.legend(); plt.grid(alpha=.3)
plt.title('PPO training loss')
plt.xlabel('training update')
plt.show()

train_result = evaluate_agent(model, train_env)
test_result = evaluate_agent(model, test_env)

report(train_result, title='Deep RL agent vs benchmark (TRAIN period)')
report(test_result, title='Deep RL agent vs benchmark (TEST period)')

json.dump({'agent':[round(x,4) for x in test_result['portfolio'].tolist()],
           'benchmark':[round(x,4) for x in test_result['benchmark'].tolist()]},
          open('dashboard/data/rl_equity.json','w'))

## 5. Read the result honestly
RL is stochastic — re-run and the number moves. It may or may not beat the benchmark
on this tiny universe. **That doesn't make it pointless.** A predictor optimizes a
proxy (guess the return); this agent optimizes the *real* objective (beat the
benchmark) and gives you honest levers to improve it — which is exactly what
tomorrow is about.

